In [17]:
import hashlib
import json
import os
import random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

In [18]:
SEMILLA = 360
random.seed(SEMILLA)
np.random.seed(SEMILLA)

CARPETA_SALIDA = "data/fuente"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

# Registro de errores insertados a propósito, para después comparar
# "errores insertados vs errores detectados" en la etapa de control de calidad
errores_insertados = []  # cada item: {"archivo":..., "tipo_error":..., "cantidad":...}


def registrar_error(archivo, tipo_error, cantidad, filas_totales):
    proporcion = round(cantidad / filas_totales, 4) if filas_totales else 0
    errores_insertados.append(
        {
            "archivo": archivo,
            "tipo_error": tipo_error,
            "cantidad": cantidad,
            "filas_totales": filas_totales,
            "proporcion": proporcion,
        }
    )


In [19]:
# =========================================================================
# 1. MAESTRO: GEOGRAFÍA (comunas, con vigencias — ojo con Ñuble)
# =========================================================================
# La Región de Ñuble se creó en 2018 separándose del Biobío: por eso varias
# comunas tienen dos filas de vigencia (antes/después del cambio de región).

comunas_maule = [
    ("07101", "Talca"), ("07102", "Constitución"), ("07103", "Curepto"),
    ("07104", "Empedrado"), ("07105", "Maule"), ("07106", "Pelarco"),
    ("07107", "Pencahue"), ("07108", "Rio Claro"), ("07109", "San Clemente"),
    ("07110", "San Rafael"), ("07201", "Cauquenes"), ("07202", "Chanco"),
    ("07203", "Pelluhue"), ("07301", "Curico"), ("07302", "Hualane"),
    ("07303", "Licanten"), ("07304", "Molina"), ("07305", "Rauco"),
    ("07306", "Romeral"), ("07307", "Sagrada Familia"), ("07308", "Teno"),
    ("07309", "Vichuquen"), ("07401", "Linares"), ("07402", "Colbun"),
    ("07403", "Longavi"), ("07404", "Parral"), ("07405", "Retiro"),
    ("07406", "San Javier"), ("07407", "Villa Alegre"), ("07408", "Yerbas Buenas"),
]
comunas_nuble = [("16101", "Chillan"), ("16102", "Bulnes"), ("16103", "Quirihue")]

filas_geo = []
for codigo, nombre in comunas_maule:
    filas_geo.append({
        "codigo_comuna": codigo, "codigo_cut_vigente": codigo, "comuna": nombre,
        "provincia": "Talca" if codigo.startswith("071") else (
            "Cauquenes" if codigo.startswith("072") else (
                "Curico" if codigo.startswith("073") else "Linares")),
        "region": "Maule", "pais": "Chile",
        "valido_desde": "1979-01-01", "valido_hasta": "", "es_actual": 1,
    })
# Comunas de Ñuble: antes de 2018 pertenecían al Biobío (región 08),
# desde 2018 son región 16. Dos filas por comuna para simular el cambio.
for codigo, nombre in comunas_nuble:
    codigo_antiguo = "08" + codigo[2:]
    filas_geo.append({
        "codigo_comuna": codigo_antiguo, "codigo_cut_vigente": codigo, "comuna": nombre,
        "provincia": "Biobio", "region": "Biobio", "pais": "Chile",
        "valido_desde": "1979-01-01", "valido_hasta": "2018-09-06", "es_actual": 0,
    })
    filas_geo.append({
        "codigo_comuna": codigo, "codigo_cut_vigente": codigo, "comuna": nombre,
        "provincia": "Diguillin", "region": "Nuble", "pais": "Chile",
        "valido_desde": "2018-09-06", "valido_hasta": "", "es_actual": 1,
    })

df_geo = pd.DataFrame(filas_geo)
df_geo.to_csv(f"{CARPETA_SALIDA}/geografia_cut.csv", index=False, encoding="utf-8")

comunas_activas = [c for _, c in comunas_maule] + [c for _, c in comunas_nuble]

In [20]:
# =========================================================================
# 2. MAESTRO: SUCURSALES v1 / v2 (cambia la zona comercial de algunas)
# =========================================================================

nombres_sucursales = [
    ("S01", "Talca Centro", "Talca"), ("S02", "Talca Mall", "Talca"),
    ("S03", "Curico Centro", "Curico"), ("S04", "Linares Centro", "Linares"),
    ("S05", "Cauquenes Centro", "Cauquenes"), ("S06", "Constitucion Plaza", "Constitución"),
    ("S07", "Molina Centro", "Molina"), ("S08", "San Javier Centro", "San Javier"),
]

sucursales_v1 = []
for cod, nombre, comuna in nombres_sucursales:
    sucursales_v1.append({
        "codigo_sucursal": cod, "sucursal": nombre, "tipo_sucursal": "Tienda fisica",
        "comuna": comuna, "zona_comercial": "Zona Sur Poniente", "version": "v1",
    })
df_suc_v1 = pd.DataFrame(sucursales_v1)
df_suc_v1.to_csv(f"{CARPETA_SALIDA}/sucursales_v1.csv", index=False, encoding="utf-8")

# v2: reorganizan zonas comerciales a mitad de año (afecta a 3 sucursales)
sucursales_v2 = [dict(s) for s in sucursales_v1]
for s in sucursales_v2:
    if s["codigo_sucursal"] in ("S01", "S02", "S06"):
        s["zona_comercial"] = "Zona Costa"
    s["version"] = "v2"
df_suc_v2 = pd.DataFrame(sucursales_v2)
df_suc_v2.to_csv(f"{CARPETA_SALIDA}/sucursales_v2.csv", index=False, encoding="utf-8")

codigos_sucursal = [s["codigo_sucursal"] for s in sucursales_v1]

In [21]:
# 3. MAESTRO: PRODUCTOS v1 / v2 + EQUIVALENCIAS (3 sistemas de codificación)
# =========================================================================

categorias = [
    ("Abarrotes", "Confites"), ("Abarrotes", "Lacteos"), ("Bebidas", "Gaseosas"),
    ("Bebidas", "Jugos"), ("Aseo", "Limpieza hogar"), ("Aseo", "Cuidado personal"),
    ("Electro", "Pequenos electrodomesticos"), ("Hogar", "Textil hogar"),
]
marcas = ["Andina", "SurChile", "Maule S.A.", "Genérica", "ImportMax"]

NUM_PRODUCTOS = 50
productos_v1, equivalencias = [], []
for i in range(1, NUM_PRODUCTOS + 1):
    sku = f"SKU-{i:04d}"
    cat, subcat = categorias[i % len(categorias)]
    precio = int(np.random.randint(990, 45000) // 10 * 10)
    costo = int(precio * np.random.uniform(0.55, 0.75))
    productos_v1.append({
        "sku_corporativo": sku, "producto": f"{subcat} {marcas[i % len(marcas)]} #{i}",
        "categoria": cat, "subcategoria": subcat, "marca": marcas[i % len(marcas)],
        "precio_lista": precio, "costo_unitario": costo, "version": "v1",
    })
    # Códigos equivalentes por sistema, cada uno con su propia convención
    codigo_pos = f"NB-{i:03d}"
    codigo_web = f"WEB-{1000 + i}"
    codigo_legado = f"NB{i:03d}"  # mismo código que POS pero SIN el guion
    equivalencias.append({
        "codigo_pos": codigo_pos, "codigo_web": codigo_web,
        "codigo_legado": codigo_legado, "sku_corporativo": sku,
    })

df_productos_v1 = pd.DataFrame(productos_v1)
df_productos_v1.to_csv(f"{CARPETA_SALIDA}/productos_v1.csv", index=False, encoding="utf-8")

# v2: suben precios ~5-10% a mitad de año y agregan 5 productos nuevos
productos_v2 = []
for p in productos_v1:
    p2 = dict(p)
    p2["precio_lista"] = int(p["precio_lista"] * np.random.uniform(1.03, 1.10) // 10 * 10)
    p2["version"] = "v2"
    productos_v2.append(p2)
for i in range(NUM_PRODUCTOS + 1, NUM_PRODUCTOS + 6):
    sku = f"SKU-{i:04d}"
    cat, subcat = categorias[i % len(categorias)]
    precio = int(np.random.randint(990, 45000) // 10 * 10)
    costo = int(precio * np.random.uniform(0.55, 0.75))
    productos_v2.append({
        "sku_corporativo": sku, "producto": f"{subcat} {marcas[i % len(marcas)]} #{i}",
        "categoria": cat, "subcategoria": subcat, "marca": marcas[i % len(marcas)],
        "precio_lista": precio, "costo_unitario": costo, "version": "v2",
    })
    equivalencias.append({
        "codigo_pos": f"NB-{i:03d}", "codigo_web": f"WEB-{1000 + i}",
        "codigo_legado": f"NB{i:03d}", "sku_corporativo": sku,
    })

df_productos_v2 = pd.DataFrame(productos_v2)
df_productos_v2.to_csv(f"{CARPETA_SALIDA}/productos_v2.csv", index=False, encoding="utf-8")

df_equivalencias = pd.DataFrame(equivalencias)
df_equivalencias.to_csv(f"{CARPETA_SALIDA}/equivalencias_productos.csv", index=False, encoding="utf-8")

codigos_pos_validos = df_equivalencias["codigo_pos"].tolist()
codigos_web_validos = df_equivalencias["codigo_web"].tolist()
codigos_legado_validos = df_equivalencias["codigo_legado"].tolist()

In [22]:
# 4. MAESTRO: CAMPAÑAS
# =========================================================================

campanas = [
    {"codigo_campana": "C01", "campana": "Verano DataRetail", "tipo_campana": "Estacional",
     "fecha_inicio": "2025-01-05", "fecha_fin": "2025-02-28", "canal_objetivo": "Todos"},
    {"codigo_campana": "C02", "campana": "CyberDay Otoño", "tipo_campana": "Evento digital",
     "fecha_inicio": "2025-05-26", "fecha_fin": "2025-05-28", "canal_objetivo": "E-commerce"},
    {"codigo_campana": "C03", "campana": "Fiestas Patrias", "tipo_campana": "Estacional",
     "fecha_inicio": "2025-09-10", "fecha_fin": "2025-09-19", "canal_objetivo": "Todos"},
    {"codigo_campana": "C04", "campana": "CyberMonday", "tipo_campana": "Evento digital",
     "fecha_inicio": "2025-11-03", "fecha_fin": "2025-11-04", "canal_objetivo": "E-commerce"},
]
df_campanas = pd.DataFrame(campanas)
df_campanas.to_csv(f"{CARPETA_SALIDA}/campanas.csv", index=False, encoding="utf-8")

In [23]:
#5. MAESTRO: CLIENTES CRM v1 / v2 (RUT en 3 formatos distintos)
# =========================================================================

nombres = ["Javiera", "Matias", "Camila", "Benjamin", "Fernanda", "Nicolas",
           "Catalina", "Diego", "Valentina", "Sebastian", "Francisca", "Tomas"]
apellidos = ["Gonzalez", "Munoz", "Rojas", "Diaz", "Perez", "Soto", "Contreras", "Silva"]


def generar_rut_valido():
    """Genera un RUT chileno válido (con dígito verificador correcto),
    calculado con el algoritmo módulo 11."""
    numero = random.randint(5_000_000, 25_000_000)
    reversed_digits = [int(d) for d in str(numero)][::-1]
    factores = [2, 3, 4, 5, 6, 7]
    suma = sum(d * factores[i % 6] for i, d in enumerate(reversed_digits))
    resto = 11 - (suma % 11)
    dv = {11: "0", 10: "K"}.get(resto, str(resto))
    return numero, dv


def formatear_rut(numero, dv, formato):
    """Formatea el mismo RUT de 3 formas distintas, tal como llegaría de
    cada sistema (con puntos y guion, sin puntos con guion, o sin nada)."""
    if formato == "con_puntos":
        s = f"{numero:,}".replace(",", ".")
        return f"{s}-{dv}"
    if formato == "sin_puntos":
        return f"{numero}-{dv}"
    return f"{numero}{dv}"  # "plano", sin separadores


NUM_CLIENTES = 300
clientes_v1 = []
ruts_clientes = []  # guardamos (numero, dv) para reutilizar en v2 y en ventas
for i in range(1, NUM_CLIENTES + 1):
    numero, dv = generar_rut_valido()
    ruts_clientes.append((numero, dv))
    formato = random.choice(["con_puntos", "sin_puntos", "plano"])
    clientes_v1.append({
        "rut": formatear_rut(numero, dv, formato),
        "nombre": random.choice(nombres),
        "apellido": random.choice(apellidos),
        "correo": f"cliente{i}@correo-demo.cl",
        "segmento": random.choice(["Frecuente", "Ocasional", "Nuevo"]),
        "version": "v1",
    })
df_clientes_v1 = pd.DataFrame(clientes_v1)
df_clientes_v1.to_csv(f"{CARPETA_SALIDA}/clientes_crm_v1.csv", index=False, encoding="utf-8")

# v2: algunos clientes cambian de segmento + se suman 40 clientes nuevos
clientes_v2 = []
for idx, c in enumerate(clientes_v1):
    c2 = dict(c)
    if random.random() < 0.15:
        c2["segmento"] = random.choice(["Frecuente", "Ocasional", "Nuevo"])
    c2["version"] = "v2"
    clientes_v2.append(c2)
for i in range(NUM_CLIENTES + 1, NUM_CLIENTES + 41):
    numero, dv = generar_rut_valido()
    ruts_clientes.append((numero, dv))
    formato = random.choice(["con_puntos", "sin_puntos", "plano"])
    clientes_v2.append({
        "rut": formatear_rut(numero, dv, formato),
        "nombre": random.choice(nombres), "apellido": random.choice(apellidos),
        "correo": f"cliente{i}@correo-demo.cl",
        "segmento": random.choice(["Frecuente", "Ocasional", "Nuevo"]), "version": "v2",
    })
df_clientes_v2 = pd.DataFrame(clientes_v2)
df_clientes_v2.to_csv(f"{CARPETA_SALIDA}/clientes_crm_v2.csv", index=False, encoding="utf-8")


In [24]:
# =========================================================================
# 6. HELPERS PARA ENSUCIAR DATOS TRANSACCIONALES
# =========================================================================

def rut_para_venta():
    """La mitad de las veces un RUT válido con dígito verificador correcto,
    la otra mitad un RUT con DV mal calculado (error típico de digitación)."""
    numero, dv = random.choice(ruts_clientes)
    if random.random() < 0.5:
        dv_malo = random.choice([d for d in "0123456789K" if d != dv])
        dv = dv_malo
    return numero, dv


def fecha_pos(fecha):
    """POS mezcla DD/MM/AAAA con AAAA/MM/DD en el mismo archivo."""
    if random.random() < 0.08:
        return fecha.strftime("%Y/%m/%d")
    return fecha.strftime("%d/%m/%Y")


def monto_pos(valor):
    """A veces el monto viene como texto con coma decimal chilena."""
    if random.random() < 0.1:
        return f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return str(valor)


def canal_pos():
    return random.choices(
        ["Tienda fisica", "TIENDA FISICA", "tienda fisica "],
        weights=[0.8, 0.12, 0.08],
    )[0]


def codigo_pos_sucio(codigo):
    if random.random() < 0.05:
        return codigo.lower() + " "  # minúscula + espacio final
    return codigo

In [25]:
# =========================================================================
# 7. TRANSACCIONAL: VENTAS POS (tienda física) — 2 semestres 2025
# =========================================================================

def generar_pos_ventas(semestre, fecha_ini, fecha_fin, n_filas):
    filas = []
    dias_rango = (fecha_fin - fecha_ini).days
    for boleta in range(1, n_filas + 1):
        fecha = fecha_ini + timedelta(days=random.randint(0, dias_rango))
        codigo = random.choice(codigos_pos_validos)
        # 2% de las líneas referencian un código que NO existe en el catálogo
        if random.random() < 0.02:
            codigo = "NB-999"
        cantidad = random.randint(1, 6)
        precio = random.choice([990, 1490, 2990, 4990, 7990, 12990, 19990, 29990])
        numero, dv = rut_para_venta()
        rut_formato = random.choice(["con_puntos", "sin_puntos", "plano"])
        filas.append({
            "nro_boleta": f"B{semestre}-{boleta:06d}",
            "linea_boleta": 1,
            "fecha": fecha_pos(fecha),
            "codigo_sucursal": random.choice(codigos_sucursal + [""]) if random.random() < 0.01 else random.choice(codigos_sucursal),
            "codigo_producto": codigo_pos_sucio(codigo),
            "canal": canal_pos(),
            "cantidad": cantidad,
            "precio_unitario": monto_pos(precio),
            "rut_cliente": formatear_rut(numero, dv, rut_formato),
        })
    df = pd.DataFrame(filas)
    # Insertamos un 1.5% de filas duplicadas exactas (error clásico de doble envío)
    n_dup = max(1, int(len(df) * 0.015))
    duplicados = df.sample(n=n_dup, random_state=SEMILLA)
    df = pd.concat([df, duplicados], ignore_index=True)
    registrar_error(f"pos_ventas_2025_S{semestre}", "filas_duplicadas", n_dup, len(df))
    registrar_error(f"pos_ventas_2025_S{semestre}", "codigo_producto_inexistente",
                     int((df["codigo_producto"].str.strip().str.upper() == "NB-999").sum()), len(df))
    return df


df_pos_s1 = generar_pos_ventas(1, datetime(2025, 1, 1), datetime(2025, 6, 30), 1800)
df_pos_s1.to_csv(f"{CARPETA_SALIDA}/pos_ventas_2025_S1.csv", index=False, sep=",", encoding="utf-8")

df_pos_s2 = generar_pos_ventas(2, datetime(2025, 7, 1), datetime(2025, 12, 31), 1900)
df_pos_s2.to_csv(f"{CARPETA_SALIDA}/pos_ventas_2025_S2.csv", index=False, sep=",", encoding="utf-8")

In [26]:
# 8. TRANSACCIONAL: PEDIDOS WEB (e-commerce / app) — separador ";" , decimal coma
# =========================================================================

def generar_web_pedidos(semestre, fecha_ini, fecha_fin, n_filas):
    filas = []
    dias_rango = (fecha_fin - fecha_ini).days
    for pedido in range(1, n_filas + 1):
        fecha = fecha_ini + timedelta(days=random.randint(0, dias_rango),
                                       hours=random.randint(0, 23), minutes=random.randint(0, 59))
        codigo = random.choice(codigos_web_validos)
        if random.random() < 0.015:
            codigo = "WEB-9999"
        cantidad = random.randint(1, 4)
        precio = random.choice([990, 1490, 2990, 4990, 7990, 12990, 19990, 29990])
        descuento_pct = random.choice([0, 0, 0, 5, 10, 15, 20])
        numero, dv = rut_para_venta()
        filas.append({
            "nro_pedido": f"W{semestre}-{pedido:06d}",
            "fecha": fecha.strftime("%Y-%m-%dT%H:%M"),
            "codigo_producto": codigo,
            "canal": random.choice(["WEB", "APP", "ecommerce"]),
            "cantidad": cantidad,
            "precio_unitario": f"{precio:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
            "descuento_pct": descuento_pct,
            "rut_cliente": formatear_rut(numero, dv, "sin_puntos"),
        })
    df = pd.DataFrame(filas)
    # 3% de los pedidos quedan con cantidad nula (error de captura del formulario)
    n_nulos = max(1, int(len(df) * 0.03))
    idx_nulos = df.sample(n=n_nulos, random_state=SEMILLA).index
    df.loc[idx_nulos, "cantidad"] = None
    registrar_error(f"web_pedidos_2025_S{semestre}", "cantidad_nula", n_nulos, len(df))
    registrar_error(f"web_pedidos_2025_S{semestre}", "codigo_producto_inexistente",
                     int((df["codigo_producto"] == "WEB-9999").sum()), len(df))
    return df


df_web_s1 = generar_web_pedidos(1, datetime(2025, 1, 1), datetime(2025, 6, 30), 950)
df_web_s1.to_csv(f"{CARPETA_SALIDA}/web_pedidos_2025_S1.csv", index=False, sep=";", encoding="utf-8")

df_web_s2 = generar_web_pedidos(2, datetime(2025, 7, 1), datetime(2025, 12, 31), 1050)
df_web_s2.to_csv(f"{CARPETA_SALIDA}/web_pedidos_2025_S2.csv", index=False, sep=";", encoding="utf-8")

In [27]:
# =========================================================================
# 9. TRANSACCIONAL: SISTEMA LEGADO 2018 (histórico, códigos sin guion)
# =========================================================================

filas_legado = []
fecha_ini, fecha_fin = datetime(2018, 1, 1), datetime(2018, 12, 31)
dias_rango = (fecha_fin - fecha_ini).days
for boleta in range(1, 700):
    fecha = fecha_ini + timedelta(days=random.randint(0, dias_rango))
    codigo = random.choice(codigos_legado_validos)
    numero, dv = rut_para_venta()
    filas_legado.append({
        "nro_boleta": f"L18-{boleta:06d}",
        "linea_boleta": 1,
        "fecha": fecha.strftime("%d/%m/%Y"),
        "codigo_sucursal": random.choice(codigos_sucursal),
        "codigo_producto": codigo,
        "canal": "Tienda fisica",
        "cantidad": random.randint(1, 5),
        "precio_unitario": random.choice([990, 1490, 2990, 4990, 7990]),
        "rut_cliente": formatear_rut(numero, dv, "plano"),
    })
df_legado = pd.DataFrame(filas_legado)
df_legado.to_csv(f"{CARPETA_SALIDA}/legado_ventas_2018.csv", index=False, encoding="utf-8")


In [28]:
# =========================================================================
# 10. TRANSACCIONAL: DEVOLUCIONES — 2 semestres
# =========================================================================

def generar_devoluciones(semestre, df_ventas_ref, n_filas):
    filas = []
    muestra = df_ventas_ref.sample(n=min(n_filas, len(df_ventas_ref)), random_state=SEMILLA + semestre)
    for _, fila in muestra.iterrows():
        filas.append({
            "nro_boleta": fila["nro_boleta"],
            "linea_boleta": fila["linea_boleta"],
            "fecha": fila["fecha"],
            "cantidad_devuelta": random.randint(1, int(fila["cantidad"]) if str(fila["cantidad"]).isdigit() else 1),
            "motivo": random.choice(["Producto defectuoso", "Arrepentimiento", "Talla/empaque incorrecto"]),
        })
    return pd.DataFrame(filas)


df_dev_s1 = generar_devoluciones(1, df_pos_s1, 120)
df_dev_s1.to_csv(f"{CARPETA_SALIDA}/devoluciones_2025_S1.csv", index=False, encoding="utf-8")

df_dev_s2 = generar_devoluciones(2, df_pos_s2, 130)
df_dev_s2.to_csv(f"{CARPETA_SALIDA}/devoluciones_2025_S2.csv", index=False, encoding="utf-8")

In [29]:
# =========================================================================
# 11. TRANSACCIONAL: STOCK DIARIO (snapshot semanal por producto x sucursal)
# =========================================================================

def generar_stock_diario(semestre, fecha_ini, fecha_fin):
    filas = []
    fecha = fecha_ini
    while fecha <= fecha_fin:
        for codigo in codigos_pos_validos:
            for suc in codigos_sucursal:
                stock = random.randint(-5, 120)  # algunos quedan negativos (error a cuarentena)
                filas.append({
                    "fecha": fecha.strftime("%d/%m/%Y"),
                    "codigo_sucursal": suc,
                    "codigo_producto": codigo,
                    "stock_cierre": stock,
                    "unidades_recibidas": random.choice([0, 0, 0, 10, 20, 50]),
                })
        fecha += timedelta(days=7)  # un snapshot semanal, no diario, para no explotar el volumen
    df = pd.DataFrame(filas)
    n_neg = int((df["stock_cierre"] < 0).sum())
    registrar_error(f"stock_diario_2025_S{semestre}", "stock_negativo", n_neg, len(df))
    return df


df_stock_s1 = generar_stock_diario(1, datetime(2025, 1, 6), datetime(2025, 6, 30))
df_stock_s1.to_csv(f"{CARPETA_SALIDA}/stock_diario_2025_S1.csv", index=False, encoding="utf-8")

df_stock_s2 = generar_stock_diario(2, datetime(2025, 7, 7), datetime(2025, 12, 29))
df_stock_s2.to_csv(f"{CARPETA_SALIDA}/stock_diario_2025_S2.csv", index=False, encoding="utf-8")

In [30]:
# =========================================================================
# 12. TABLA DE ERRORES INSERTADOS (para comparar luego con los detectados)
# =========================================================================

df_errores = pd.DataFrame(errores_insertados)
df_errores.to_csv(f"{CARPETA_SALIDA}/_errores_insertados.csv", index=False, encoding="utf-8")
print("\n=== Errores insertados a propósito (por archivo) ===")
print(df_errores.to_string(index=False))


=== Errores insertados a propósito (por archivo) ===
             archivo                  tipo_error  cantidad  filas_totales  proporcion
  pos_ventas_2025_S1            filas_duplicadas        27           1827      0.0148
  pos_ventas_2025_S1 codigo_producto_inexistente        41           1827      0.0224
  pos_ventas_2025_S2            filas_duplicadas        28           1928      0.0145
  pos_ventas_2025_S2 codigo_producto_inexistente        41           1928      0.0213
 web_pedidos_2025_S1               cantidad_nula        28            950      0.0295
 web_pedidos_2025_S1 codigo_producto_inexistente        16            950      0.0168
 web_pedidos_2025_S2               cantidad_nula        31           1050      0.0295
 web_pedidos_2025_S2 codigo_producto_inexistente        19           1050      0.0181
stock_diario_2025_S1              stock_negativo       450          11440      0.0393
stock_diario_2025_S2              stock_negativo       439          11440      0.0384


In [31]:
# =========================================================================
# 13. MANIFEST.JSON — hash SHA-256 de cada archivo generado, para verificar
#     integridad cada vez que el notebook reutiliza estos datos
# =========================================================================

manifest = {"semilla": SEMILLA, "generado_en": datetime.now().isoformat(), "archivos": {}}
for nombre_archivo in sorted(os.listdir(CARPETA_SALIDA)):
    ruta = os.path.join(CARPETA_SALIDA, nombre_archivo)
    if nombre_archivo == "manifest.json":
        continue
    with open(ruta, "rb") as f:
        contenido = f.read()
    sha256 = hashlib.sha256(contenido).hexdigest()
    manifest["archivos"][nombre_archivo] = {
        "sha256": sha256,
        "bytes": len(contenido),
    }

with open(f"{CARPETA_SALIDA}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"\n¡Listo! Se generaron {len(manifest['archivos'])} archivos en {CARPETA_SALIDA}/")
print("manifest.json creado con el hash SHA-256 de cada uno.")


¡Listo! Se generaron 19 archivos en data/fuente/
manifest.json creado con el hash SHA-256 de cada uno.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')